In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [3]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## Text cleaning methods:
Creating a function to convert the raw text data into cleaner text form by:

1. Lowercasing the texts
2. Removing any kind of special characters
3. Reducing multiple white spaces(if there) into one.

In [4]:
import re

def text_clean(text):
  lower_text = text.lower()
  simple_text = re.sub(r'[^a-z0-9\s]', '', lower_text)
  cleaned_text = re.sub(r'\s+', ' ', simple_text)
  return cleaned_text

In [5]:
#applying function to all the columns in both train and test
cols = ['prompt' , 'A', 'B' , 'C' , 'D' , 'E']

for col in cols:
    train[col +'_clean'] = train[col].apply(text_clean)
    test[col +'_clean'] = test[col].apply(text_clean)

In [6]:
train.head(1)

,id,prompt,A,B,C,D,E,answer,prompt_clean,A_clean,B_clean,C_clean,D_clean,E_clean
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...


In [7]:
train.shape

(2000, 14)

In [8]:
x = train.copy()

## Applying train test split on the dataset

In [9]:
from sklearn.model_selection import train_test_split

x_train, x_test = train_test_split(x, test_size=0.2, random_state=42)

In [10]:
x_train.shape, x_test.shape

((1600, 14), (400, 14))

In [11]:
#function to build prompt and option pairs

def build_pairs(df):
    rows = []
    for i in df.index:
        prompt = df.loc[i, 'prompt_clean']
        options = ['A', 'B', 'C', 'D', 'E']
        for opt in options:
            text = prompt + ' ' + df.loc[i, f'{opt}_clean']
            label = 0
            if df.loc[i, 'answer'] == opt:
                label = 1

            rows.append({'id': df.loc[i, 'id'],
                         'option': opt,
                         'text': text,
                         'label': label})
    return pd.DataFrame(rows)

train_pairs = build_pairs(x_train)
val_pairs = build_pairs(x_test)

print(train_pairs.shape, val_pairs.shape)

(8000, 4) (2000, 4)


In [12]:
train_pairs.iloc[0:5]

,id,option,text,label
0,969,A,pick the best possible answer what is the prop...,1
1,969,B,pick the best possible answer what is the prop...,0
2,969,C,pick the best possible answer what is the prop...,0
3,969,D,pick the best possible answer what is the prop...,0
4,969,E,pick the best possible answer what is the prop...,0


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1,3),
                    max_features=30000,
                        min_df = 2,
                        sublinear_tf = True)

In [14]:
x_train_tfidf = tfidf.fit_transform(train_pairs['text'])
x_test_tfidf = tfidf.transform(val_pairs['text'])

y_train_label = train_pairs['label']
y_test_label = val_pairs['label']

x_train_tfidf.shape

(8000, 28450)

In [15]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=5000 ,class_weight = 'balanced', random_state = 42)
lr.fit(x_train_tfidf, y_train_label)

LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42)

In [16]:
def map_at3(prediction,correct):  #function to calculate MAP@3 score
  score = 0
  for pred, x in zip(prediction, correct):
    if x in pred:
      k = pred.index(x) + 1
      score += 1/k
  return round(score/len(correct) ,4)

In [17]:
val_pairs['prob'] = lr.predict_proba(x_test_tfidf)[:, 1]

def get_top3_predictions(pairs_df):
    predictions = []
    qids = []
    for qid, group in pairs_df.groupby('id'):
        top3 = group.sort_values('prob', ascending=False)['option'].tolist()[:3]
        predictions.append(top3)
        qids.append(qid)
    return predictions, qids

val_top3, val_qids = get_top3_predictions(val_pairs)
actual_answers = x_test.set_index('id').loc[val_qids, 'answer'].tolist()

score = map_at3(val_top3, actual_answers)
print("TF-IDF + Logistic Regression MAP@3:", score)

TF-IDF + Logistic Regression MAP@3: 0.9938


In [18]:
val_pairs.iloc[:10]

,id,option,text,label,prob
0,1861,A,select the most accurate option what is the pr...,0,0.275765
1,1861,B,select the most accurate option what is the pr...,0,0.236268
2,1861,C,select the most accurate option what is the pr...,0,0.418122
3,1861,D,select the most accurate option what is the pr...,1,0.648659
4,1861,E,select the most accurate option what is the pr...,0,0.434574
5,354,A,pick the best possible answer what is the most...,0,0.259075
6,354,B,pick the best possible answer what is the most...,0,0.330388
7,354,C,pick the best possible answer what is the most...,0,0.234732
8,354,D,pick the best possible answer what is the most...,0,0.298763
9,354,E,pick the best possible answer what is the most...,1,0.702472


## Generate Kaggle Submission

Same pipeline as validation, applied to the test set (no answers available, so no
MAP@3 here - just predictions). Wrapped into reusable functions so the same pattern
can be reused later for the BERT and LoRA models.

In [19]:
def build_test_pairs(df): #Creates one row per (question, option) pair for the test set.
    rows = []
    for i in df.index:
        prompt =df.loc[i,'prompt_clean']
        
        for opt in ['A','B','C','D','E']:
            text= prompt + ' ' + df.loc[i, f'{opt}_clean']
            rows.append({'id': df.loc[i, 'id'], 
                         'option': opt, 
                         'text': text})
            
    return pd.DataFrame(rows)
    

In [20]:
test_pairs = build_test_pairs(test)
test_features = tfidf.transform(test_pairs["text"])

probabilities = lr.predict_proba(test_features)[:, 1]

test_pairs["prob"] = probabilities

In [21]:
predictions = []

for question_id, group in test_pairs.groupby("id"):
    sorted_group = group.sort_values("prob",ascending=False)

    top3_options = sorted_group["option"].tolist()[:3]

    top3_prediction = " ".join(top3_options)

    predictions.append(top3_prediction)

In [22]:
submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
submission["prediction"] = predictions

submission.to_csv("/kaggle/working/submission.csv",index=False)

print("Submission shape:", submission.shape)
submission.head()

Submission shape: (500, 3)


,ID,Prediction,prediction
0,1,A B C,A E B
1,2,A B C,B C E
2,3,A B C,B E D
3,4,A B C,E C D
4,5,A B C,C A D


**Kaggle score :** 0.75103